# Live-to-Canonical Station Mapping – Objective & Plan

## Objective
Retrieve live BIXI station information from `info_url` and map every live station to the cleaned canonical station mapping built in `01_dataeng_stations.ipynb`.

## Plan (task-gated)
1. **Task 1:** Retrieve and flatten live station feed into a DataFrame.
2. **Task 2:** Load canonical mapping artifact exported from the cleaning notebook.
3. **Task 3:** Match live stations to canonical stations using normalized name + coordinate proximity rules.
4. **Task 4:** Report match coverage, unmatched stations, and likely conflict cases.

> Following your instruction: each task will be executed and checked before moving to the next.

In [1]:
import requests

In [2]:
# URLs
info_url   = "https://gbfs.velobixi.com/gbfs/2-2/en/station_information.json"

# Load JSON
info_data   = requests.get(info_url).json()
info_data

{'last_updated': 1772697811,
 'ttl': 10,
 'version': '2.2',
 'data': {'stations': [{'station_id': '3',
    'external_id': '0b0fe114-08f3-11e7-a1cb-3863bb33a4e4',
    'name': 'Clark / Ontario',
    'short_name': '6003',
    'lat': 45.510587509716636,
    'lon': -73.56684565544128,
    'rental_methods': ['CREDITCARD', 'KEY'],
    'capacity': 19,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '15',
    'external_id': '0b0ffa68-08f3-11e7-a1cb-3863bb33a4e4',
    'name': 'Métro Berri-UQAM (St-Denis / de Maisonneuve)',
    'short_name': '6015',
    'lat': 45.51425199763688,
    'lon': -73.56150217976847,
    'rental_methods': ['CREDITCARD', 'KEY'],
    'capacity': 15,
    'electric_bike_surcharge_waiver': False,
    'is_charging': False,
    'eightd_has_key_dispenser': False,
    'has_kiosk': True},
   {'station_id': '17',
    'external_id': '0b0ffd3a-08f3-11e7-a1cb-3863bb33a4e4',
    'nam

In [3]:
# Task 1: flatten live station feed
import pandas as pd

live_stations_df = pd.DataFrame(info_data['data']['stations']).copy()

required_live_cols = ['station_id', 'name', 'lat', 'lon']
missing_live_cols = [c for c in required_live_cols if c not in live_stations_df.columns]

print(f"Live stations retrieved: {len(live_stations_df):,}")
print(f"Columns available: {len(live_stations_df.columns)}")
if missing_live_cols:
    print(f"Missing required live columns: {missing_live_cols}")
else:
    print("Required live columns present: station_id, name, lat, lon")

live_stations_df[['station_id', 'name', 'lat', 'lon']].head(10)

Live stations retrieved: 240
Columns available: 12
Required live columns present: station_id, name, lat, lon


,station_id,name,lat,lon
0,3,Clark / Ontario,45.510588,-73.566846
1,15,Métro Berri-UQAM (St-Denis / de Maisonneuve),45.514252,-73.561502
2,17,Marché St-Jacques (Atateken),45.520666,-73.563915
3,19,Métro Sherbrooke (de Rigaud / Berri),45.518143,-73.568004
4,24,Notre-Dame / St-Gabriel,45.507118,-73.555049
5,25,de la Commune / Place Jacques-Cartier,45.507610,-73.551836
6,26,de Maisonneuve / Mansfield (sud),45.502054,-73.573465
7,31,Métro Place-d'Armes (St-Urbain / Viger),45.506323,-73.559699
8,34,Viger / Chenneville,45.505312,-73.560891
9,35,de la Commune / St-Sulpice,45.504242,-73.553469


In [4]:
# Task 2: load canonical mapping artifact from cleaning notebook
from pathlib import Path

mapping_artifact_path = Path("data/silver/station_cleaning/station_canonical_mapping.parquet")

if not mapping_artifact_path.exists():
    raise FileNotFoundError(f"Missing artifact: {mapping_artifact_path}")

canonical_mapping_df = pd.read_parquet(mapping_artifact_path)

required_map_cols = [
    'canonical_station_id',
    'canonical_lat',
    'canonical_lon',
    'normalized_name',
    'coord_key',
]
missing_map_cols = [c for c in required_map_cols if c not in canonical_mapping_df.columns]

print(f"Canonical mapping rows: {len(canonical_mapping_df):,}")
print(f"Canonical stations: {canonical_mapping_df['canonical_station_id'].nunique():,}")
if missing_map_cols:
    print(f"Missing required mapping columns: {missing_map_cols}")
else:
    print("Required mapping columns present")

canonical_mapping_df[required_map_cols].head(10)

Canonical mapping rows: 1,922
Canonical stations: 1,577
Required mapping columns present


,canonical_station_id,canonical_lat,canonical_lon,normalized_name,coord_key
0,STN_0001,45.519409,-73.586853,du mont-royal / clark,"45.519409,-73.586853"
1,STN_0002,45.532219,-73.575432,marquette / du mont-royal,"45.532219,-73.575432"
2,STN_0002,45.532219,-73.575432,marquette / du mont-royal (sud),"45.532078,-73.575142"
3,STN_0003,45.527153,-73.589439,laurier / st-denis,"45.527153,-73.589439"
4,STN_0004,45.515228,-73.575096,des pins / st-laurent,"45.515228,-73.575096"
5,STN_0005,45.502293,-73.573303,de maisonneuve / mansfield (sud-est),"45.502293,-73.573303"
6,STN_0005,45.502293,-73.573303,de maisonneuve / mansfield (sud),"45.502052,-73.573463"
7,STN_0005,45.502293,-73.573303,de maisonneuve / mansfield (nord),"45.502384,-73.573341"
8,STN_0005,45.502293,-73.573303,rem mcgill (de maisonneuve / mansfield),"45.502293,-73.573303"
9,STN_0006,45.500381,-73.575073,métro peel (de maisonneuve / stanley),"45.500381,-73.575073"


In [6]:
# Task 3 + Task 4: map live stations to canonical stations and report coverage
import numpy as np


def normalize_station_name(text: str) -> str:
    if text is None:
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*/\s*", " / ", text)
    text = re.sub(r"\s*-\s*", "-", text)
    return text.strip()


def haversine_km_vec(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


MAX_NEAREST_KM = 0.05
MAX_NAME_COORD_KM = 0.20

live_map_df = live_stations_df.copy()
live_map_df['normalized_name'] = live_map_df['name'].map(normalize_station_name)
live_map_df['coord_key'] = live_map_df['lat'].map(lambda v: f"{float(v):.6f}") + "," + live_map_df['lon'].map(lambda v: f"{float(v):.6f}")

canon_unique_df = (
    canonical_mapping_df[['canonical_station_id', 'canonical_lat', 'canonical_lon', 'normalized_name', 'coord_key']]
    .drop_duplicates()
    .copy()
)

# Stage A: exact coordinate key
coord_lookup_df = canon_unique_df[['coord_key', 'canonical_station_id']].drop_duplicates('coord_key')
live_mapped_df = live_map_df.merge(coord_lookup_df, how='left', on='coord_key')
live_mapped_df['match_method'] = np.where(live_mapped_df['canonical_station_id'].notna(), 'exact_coord_key', None)
live_mapped_df['match_distance_km'] = np.where(live_mapped_df['canonical_station_id'].notna(), 0.0, np.nan)

# Stage B: normalized name exact + nearest coord within threshold
unmatched_idx = live_mapped_df[live_mapped_df['canonical_station_id'].isna()].index.tolist()
if unmatched_idx:
    by_name = {k: v for k, v in canon_unique_df.groupby('normalized_name')}
    for idx in unmatched_idx:
        row = live_mapped_df.loc[idx]
        name = row['normalized_name']
        if name not in by_name:
            continue
        cand = by_name[name]
        dists = haversine_km_vec(
            row['lat'], row['lon'],
            cand['canonical_lat'].to_numpy(), cand['canonical_lon'].to_numpy()
        )
        min_pos = int(np.argmin(dists))
        min_dist = float(dists[min_pos])
        if min_dist <= MAX_NAME_COORD_KM:
            live_mapped_df.at[idx, 'canonical_station_id'] = cand.iloc[min_pos]['canonical_station_id']
            live_mapped_df.at[idx, 'match_method'] = 'exact_normalized_name_nearest_coord'
            live_mapped_df.at[idx, 'match_distance_km'] = min_dist

# Stage C: nearest canonical station within MAX_NEAREST_KM
unmatched_idx = live_mapped_df[live_mapped_df['canonical_station_id'].isna()].index.tolist()
if unmatched_idx:
    canon_station_centers = (
        canon_unique_df[['canonical_station_id', 'canonical_lat', 'canonical_lon']]
        .drop_duplicates('canonical_station_id')
        .reset_index(drop=True)
    )
    center_lats = canon_station_centers['canonical_lat'].to_numpy()
    center_lons = canon_station_centers['canonical_lon'].to_numpy()

    for idx in unmatched_idx:
        row = live_mapped_df.loc[idx]
        dists = haversine_km_vec(row['lat'], row['lon'], center_lats, center_lons)
        min_pos = int(np.argmin(dists))
        min_dist = float(dists[min_pos])
        if min_dist <= MAX_NEAREST_KM:
            live_mapped_df.at[idx, 'canonical_station_id'] = canon_station_centers.iloc[min_pos]['canonical_station_id']
            live_mapped_df.at[idx, 'match_method'] = 'nearest_canonical_within_0.05km'
            live_mapped_df.at[idx, 'match_distance_km'] = min_dist

# Coverage report
mapped_count = int(live_mapped_df['canonical_station_id'].notna().sum())
total_live = int(len(live_mapped_df))
unmapped_count = total_live - mapped_count
coverage_pct = 100.0 * mapped_count / total_live if total_live else 0.0

match_method_counts_df = (
    live_mapped_df
    .assign(match_method=live_mapped_df['match_method'].fillna('unmatched'))
    .groupby('match_method', as_index=False)
    .agg(station_count=('station_id', 'count'))
    .sort_values('station_count', ascending=False)
)

unmatched_live_stations_df = (
    live_mapped_df[live_mapped_df['canonical_station_id'].isna()]
    [['station_id', 'name', 'normalized_name', 'lat', 'lon']]
    .sort_values('name')
)

print(f"Live stations total: {total_live}")
print(f"Mapped stations: {mapped_count}")
print(f"Unmapped stations: {unmapped_count}")
print(f"Coverage: {coverage_pct:.2f}%")

print("\nMatch method breakdown:")
display(match_method_counts_df)

if unmapped_count > 0:
    print("\nUnmapped live stations (first 30):")
    display(unmatched_live_stations_df.head(30))
else:
    print("All live stations mapped successfully.")

live_to_canonical_mapping_df = live_mapped_df[['station_id', 'name', 'normalized_name', 'lat', 'lon', 'canonical_station_id', 'match_method', 'match_distance_km']].copy()
display(live_to_canonical_mapping_df.head(30))

Live stations total: 240
Mapped stations: 240
Unmapped stations: 0
Coverage: 100.00%

Match method breakdown:


,match_method,station_count
1,exact_normalized_name_nearest_coord,227
0,exact_coord_key,12
2,nearest_canonical_within_0.05km,1


All live stations mapped successfully.


,station_id,name,normalized_name,lat,lon,canonical_station_id,match_method,match_distance_km
0,3,Clark / Ontario,clark / ontario,45.510588,-73.566846,STN_0359,exact_normalized_name_nearest_coord,0.011191
1,15,Métro Berri-UQAM (St-Denis / de Maisonneuve),métro berri-uqam (st-denis / de maisonneuve),45.514252,-73.561502,STN_0873,exact_normalized_name_nearest_coord,0.017155
2,17,Marché St-Jacques (Atateken),marché st-jacques (atateken),45.520666,-73.563915,STN_0151,exact_normalized_name_nearest_coord,0.000363
3,19,Métro Sherbrooke (de Rigaud / Berri),métro sherbrooke (de rigaud / berri),45.518143,-73.568004,STN_0012,exact_normalized_name_nearest_coord,0.000263
4,24,Notre-Dame / St-Gabriel,notre-dame / st-gabriel,45.507118,-73.555049,STN_1137,exact_normalized_name_nearest_coord,0.000261
5,25,de la Commune / Place Jacques-Cartier,de la commune / place jacques-cartier,45.507610,-73.551836,STN_0018,exact_normalized_name_nearest_coord,0.000157
6,26,de Maisonneuve / Mansfield (sud),de maisonneuve / mansfield (sud),45.502054,-73.573465,STN_0005,exact_normalized_name_nearest_coord,0.029447
7,31,Métro Place-d'Armes (St-Urbain / Viger),métro place-d'armes (st-urbain / viger),45.506323,-73.559699,STN_0293,exact_normalized_name_nearest_coord,0.002602
8,34,Viger / Chenneville,viger / chenneville,45.505312,-73.560891,STN_0432,exact_normalized_name_nearest_coord,0.000206
9,35,de la Commune / St-Sulpice,de la commune / st-sulpice,45.504242,-73.553469,STN_0031,exact_normalized_name_nearest_coord,0.000200
